### Packages installation

In [ ]:
pip install polars pandas duckdb pyspark faker deltalake memory_profiler pyarrow

### Spark queries running

In [ ]:
import time
import pandas as pd
from pyspark.sql import SparkSession


def measure_time(query):
    start = time.perf_counter()
    _ = query()
    elapsed = time.perf_counter() - start
    return elapsed


def benchmark_scalability_spark(instances: int = 2, cores: int = 2):
    results = []

    spark = (
        SparkSession.builder.appName("spark-benchmark")
        .config("spark.dynamicAllocation.enabled", "false")
        .config("spark.master", "yarn")
        .config("spark.executor.instances", f"{instances}")
        .config("spark.executor.cores", f"{cores}")
        .getOrCreate()
    )

    td = spark.sparkContext.defaultParallelism
    social_media = "gs://tbd-2025z-304021-dataproc-temp/social_media_data.parquet"
    users = "gs://tbd-2025z-304021-dataproc-temp/users.parquet"

    spark.conf.set("spark.sql.shuffle.partitions", td)
    df_spark = spark.read.parquet(social_media).repartition(td)
    df_spark_users = spark.read.parquet(users).repartition(td)
    df_spark.createOrReplaceTempView("posts")
    df_spark_users.createOrReplaceTempView("users")

    def query_A_spark_repartitioned():
        return df_spark.groupBy("location").avg("likes").count()

    def query_B_spark_repartitioned():
        return spark.sql(
            """
            SELECT COUNT(*) FROM (SELECT *, AVG(likes) OVER ( PARTITION BY user_id
                ORDER BY timestamp ROWS BETWEEN 2 PRECEDING AND CURRENT ROW ) AS avg_last_3 FROM posts)
            WHERE avg_last_3 > 5000"""
        ).collect()[0][0]

    def query_C_spark_repartitioned():
        return df_spark.join(df_spark_users, on="user_id").where("age >= 25").count()

    local_queries = {
        "A spark": query_A_spark_repartitioned,
        "B spark": query_B_spark_repartitioned,
        "C spark": query_C_spark_repartitioned,
    }
    for query_name, query_fn in local_queries.items():
        query_fn()

        # benchmark
        t = measure_time(query_fn)

        results.append(
            {
                "Threads": td,
                "Query": query_name,
                "Time [s]": t,
            }
        )

    spark.stop()
    return pd.DataFrame(results)


df_spark_results = benchmark_scalability_spark()
print(df_spark_results)